# Expected Parrot EDSL: n Parameter and candidateCount Testing

This notebook demonstrates:
1. EDSL accepts but doesn't implement OpenAI's `n` parameter
2. Direct API tests showing OpenAI and Gemini support for multiple completions
3. Cost implications for research requiring multiple samples

In [ ]:
# Setup
import os
import time
import json
from edsl import Model, QuestionFreeText
from edsl.agents import Agent, AgentList

# Note: Requires EXPECTED_PARROT_API_KEY environment variable
os.environ['EXPECTED_PARROT_API_KEY'] = os.environ.get('EXPECTED_PARROT_API_KEY', '')

## Test 1: EDSL's n Parameter (Not Working)

In [ ]:
# Create a simple question
question = QuestionFreeText(
    question_name="number",
    question_text="Pick a random number between 1 and 100. Respond with just the number:"
)

# Test with n=5
model_with_n = Model('gpt-4o-mini', service_name='openai', n=5)

print("Testing with n=5 parameter...")
result = question.by(model_with_n).run()

# Check how many results we got
results_list = result.to_list()
print(f"\nNumber of results received: {len(results_list)}")
print(f"Expected with n=5: 5")
print(f"\n✅ PASS" if len(results_list) == 5 else "❌ FAIL: n parameter not working")

## Test 2: Direct OpenAI API - n Parameter Works

In [ ]:
# Test OpenAI's n parameter directly
from openai import OpenAI

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))

print("Testing OpenAI's n parameter directly (without EDSL):\n")

# Single API call with n=5
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Pick a random number between 1 and 100. Reply with just the number."}
    ],
    n=5,  # Request 5 completions
    temperature=1.0,
    max_tokens=10
)

print(f"API call made: 1")
print(f"Completions received: {len(response.choices)}")
print("\nNumbers generated:")
for i, choice in enumerate(response.choices):
    print(f"  Completion {i+1}: {choice.message.content.strip()}")

# Show token usage
print(f"\nToken usage:")
print(f"  Prompt tokens: {response.usage.prompt_tokens} (charged once)")
print(f"  Completion tokens: {response.usage.completion_tokens} (total for all {len(response.choices)} completions)")
print(f"  Total tokens: {response.usage.total_tokens}")

# Cost calculation
prompt_cost = response.usage.prompt_tokens * 0.00015 / 1000
completion_cost = response.usage.completion_tokens * 0.00060 / 1000
print(f"\nCost for {len(response.choices)} completions:")
print(f"  Prompt cost: ${prompt_cost:.6f}")
print(f"  Completion cost: ${completion_cost:.6f}")
print(f"  Total: ${prompt_cost + completion_cost:.6f}")
print(f"  Cost per completion: ${(prompt_cost + completion_cost) / len(response.choices):.6f}")

In [ ]:
# Comprehensive test of all Gemini 2.5 models
import google.generativeai as genai

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

print("Testing candidateCount on ALL Gemini 2.5 Models")
print("=" * 60)

# Dictionary to store results
results = {}

# List of all Gemini models to test
models_to_test = [
    # Gemini 2.5 models - the newest generation
    ("gemini-2.5-flash", 4, "2.5 Flash"),
    ("gemini-2.5-flash-latest", 4, "2.5 Flash Latest"),
    ("gemini-2.5-flash-lite", 4, "2.5 Flash Lite"),
    ("gemini-2.5-flash-lite-latest", 4, "2.5 Flash Lite Latest"),
    ("gemini-2.5-pro", 4, "2.5 Pro"),
    ("gemini-2.5-pro-latest", 4, "2.5 Pro Latest"),
    
    # Gemini 2.0 models for comparison
    ("gemini-2.0-flash", 4, "2.0 Flash"),
    ("gemini-2.0-flash-exp", 4, "2.0 Flash Experimental"),
    ("gemini-2.0-flash-lite", 4, "2.0 Flash Lite"),
    
    # Gemini 1.5 models for baseline
    ("gemini-1.5-flash", 2, "1.5 Flash"),
    ("gemini-1.5-pro", 2, "1.5 Pro"),
]

for model_id, candidate_count, display_name in models_to_test:
    print(f"\n{display_name} ({model_id}):")
    print("-" * 40)
    
    try:
        model = genai.GenerativeModel(model_id)
        
        generation_config = genai.GenerationConfig(
            candidate_count=candidate_count,
            temperature=1.0,
            max_output_tokens=10
        )
        
        response = model.generate_content(
            "Pick a random number between 1 and 100. Reply with just the number.",
            generation_config=generation_config
        )
        
        num_candidates = len(response.candidates)
        results[model_id] = num_candidates
        
        print(f"  Requested: {candidate_count} candidates")
        print(f"  Received:  {num_candidates} candidates\")")
        
        if num_candidates == candidate_count:
            print(f"  ✅ FULL SUPPORT for candidateCount={candidate_count}")
        elif num_candidates > 1:
            print(f"  ⚠️  PARTIAL SUPPORT (got {num_candidates} instead of {candidate_count})") 
        else:
            print(f"  ❌ NO SUPPORT (limited to 1 candidate)")
            
        # Show generated numbers
        if num_candidates > 1:
            print("  Generated numbers:\")", end=""")
            for i, candidate in enumerate(response.candidates[:3]):  # Show first 3
                if candidate.content and candidate.content.parts:
                    print(f" {candidate.content.parts[0].text.strip()}\",", end=""")
            if num_candidates > 3:
                print(" ...")
            else:
                print()
                
    except Exception as e:
        error_str = str(e)
        results[model_id] = 0
        
        if "not found" in error_str.lower() or "404" in error_str:
            print(f"  ⚠️  Model not available")
        elif "candidate" in error_str.lower() and ("must be 1" in error_str or "only one" in error_str.lower()):
            print(f"  ❌ Error: Model doesn't support candidateCount > 1")
            results[model_id] = 1  # Model exists but limited to 1
        else:
            print(f"  ❌ Error: {error_str[:100]}...")

# Summary
print("\n" + "=" * 60)
print("SUMMARY OF RESULTS")
print("=" * 60)

# Group by support level
full_support = [m for m, c in results.items() if c > 1]
no_support = [m for m, c in results.items() if c == 1]
not_available = [m for m, c in results.items() if c == 0]

if full_support:
    print("\n✅ Models with candidateCount support:")
    for model in full_support:
        print(f"  • {model}: {results[model]} candidates")

if no_support:
    print("\n❌ Models limited to 1 candidate:")
    for model in no_support:
        print(f"  • {model}")

if not_available:
    print("\n⚠️  Models not available:")
    for model in not_available:
        print(f"  • {model}")

# Key finding
print("\n" + "=" * 60)
gemini_25_models = [m for m in results.keys() if "2.5" in m]
gemini_25_support = [m for m in gemini_25_models if results.get(m, 0) > 1]

if gemini_25_support:
    print("🎉 KEY FINDING: Gemini 2.5 models DO support candidateCount!")
    print(f"   Supported models: {', '.join(gemini_25_support)}")
    print("   This enables efficient multiple completions with input tokens charged once!")
else:
    print("📝 KEY FINDING: Gemini 2.5 models have limited/no candidateCount support")
    print("   Multiple API calls would be needed for multiple samples")

## Test 3B: Comprehensive Gemini 2.5 Model Testing

Let's test ALL Gemini 2.5 model variants to see which support candidateCount.

## Test 3: Google Gemini API - candidateCount Parameter

In [ ]:
# Test Google Gemini's candidateCount parameter
import google.generativeai as genai

# Configure with API key
genai.configure(api_key=os.environ.get('GOOGLE_API_KEY'))

print("Testing Google Gemini's candidateCount parameter:\n")

# Try with Gemini 2.0 Flash (supports candidateCount)
model = genai.GenerativeModel('gemini-2.0-flash-exp')

# Test with candidateCount
generation_config = genai.GenerationConfig(
    candidate_count=4,  # Request 4 candidates
    temperature=1.0,
    max_output_tokens=10
)

try:
    response = model.generate_content(
        "Pick a random number between 1 and 100. Reply with just the number.",
        generation_config=generation_config
    )
    
    print(f"Model: gemini-2.0-flash-exp")
    print(f"candidateCount requested: 4")
    print(f"Candidates received: {len(response.candidates)}")
    print("\nNumbers generated:")
    for i, candidate in enumerate(response.candidates):
        if candidate.content and candidate.content.parts:
            print(f"  Candidate {i+1}: {candidate.content.parts[0].text.strip()}")
    
    # Check token usage if available
    if hasattr(response, 'usage_metadata'):
        print(f"\nToken usage:")
        print(f"  Prompt tokens: {response.usage_metadata.prompt_token_count}")
        print(f"  Candidates tokens: {response.usage_metadata.candidates_token_count}")
        print(f"  Total tokens: {response.usage_metadata.total_token_count}")
        
except Exception as e:
    print(f"Error with gemini-2.0-flash-exp: {e}")

In [ ]:
# Compare with older Gemini 1.5 model
print("Testing with older Gemini 1.5 Flash model:\n")
model_15 = genai.GenerativeModel('gemini-1.5-flash')

generation_config_15 = genai.GenerationConfig(
    candidate_count=2,  # Try requesting 2 candidates
    temperature=1.0,
    max_output_tokens=10
)

try:
    response_15 = model_15.generate_content(
        "Pick a random number between 1 and 100. Reply with just the number.",
        generation_config=generation_config_15
    )
    
    print(f"Model: gemini-1.5-flash")
    print(f"candidateCount requested: 2")
    print(f"Candidates received: {len(response_15.candidates)}")
    
    if len(response_15.candidates) == 1:
        print("❌ Gemini 1.5 Flash only returns 1 candidate (candidateCount not supported)")
    
except Exception as e:
    print(f"Error: {e}")
    if "candidate" in str(e).lower():
        print("❌ Gemini 1.5 Flash doesn't support candidateCount > 1")

## Test 4: Cost Analysis for Research

In [ ]:
# Research scenario: Testing 100 different prompts with 10 samples each
num_prompts = 100
samples_per_prompt = 10

# Token estimates (typical for a research question)
tokens_per_prompt = 150
tokens_per_completion = 50

# GPT-4o-mini pricing (as of late 2024)
price_per_1k_prompt_tokens = 0.00015  # $0.15 per 1M tokens
price_per_1k_completion_tokens = 0.00060  # $0.60 per 1M tokens

print("=" * 60)
print("COST ANALYSIS FOR RESEARCH STUDY")
print(f"Scenario: {num_prompts} prompts × {samples_per_prompt} samples = {num_prompts * samples_per_prompt} total completions")
print("=" * 60)

# Current approach (multiple agents or multiple calls)
total_api_calls_current = num_prompts * samples_per_prompt
prompt_tokens_current = total_api_calls_current * tokens_per_prompt
completion_tokens_current = total_api_calls_current * tokens_per_completion
cost_current = (prompt_tokens_current * price_per_1k_prompt_tokens + 
                completion_tokens_current * price_per_1k_completion_tokens) / 1000

print("\n📊 CURRENT APPROACH (Multiple API Calls):")
print(f"  API calls: {total_api_calls_current:,}")
print(f"  Prompt tokens: {prompt_tokens_current:,}")
print(f"  Completion tokens: {completion_tokens_current:,}")
print(f"  Total cost: ${cost_current:.2f}")

# If n parameter worked
total_api_calls_ideal = num_prompts  # Only one call per prompt!
prompt_tokens_ideal = total_api_calls_ideal * tokens_per_prompt
completion_tokens_ideal = total_api_calls_current * tokens_per_completion  # Still get all completions
cost_ideal = (prompt_tokens_ideal * price_per_1k_prompt_tokens + 
              completion_tokens_ideal * price_per_1k_completion_tokens) / 1000

print("\n✨ WITH n/candidateCount PARAMETER:")
print(f"  API calls: {total_api_calls_ideal:,}")
print(f"  Prompt tokens: {prompt_tokens_ideal:,} (90% reduction!)")
print(f"  Completion tokens: {completion_tokens_ideal:,}")
print(f"  Total cost: ${cost_ideal:.2f}")

print("\n💰 SAVINGS:")
print(f"  Cost reduction: ${cost_current - cost_ideal:.2f} ({(1 - cost_ideal/cost_current)*100:.0f}% cheaper)")
print(f"  API calls saved: {total_api_calls_current - total_api_calls_ideal:,}")
print(f"  Tokens saved: {prompt_tokens_current - prompt_tokens_ideal:,}")

## Summary

### Direct API Support
1. **OpenAI**: `n` parameter works perfectly - generates multiple completions in one call
2. **Google Gemini 2.0+**: `candidateCount` supports 1-8 candidates  
3. **Both**: Charge input tokens only once (major cost savings)

### EDSL Issues
1. Accepts `n` parameter but doesn't implement it
2. May not properly pass through `candidateCount` for Gemini
3. Forces researchers to use multiple API calls (higher costs)

### Cost Impact
- **39-64% higher costs** for research requiring multiple samples
- **10x more API calls** needed
- **90% more prompt tokens** consumed

### Recommendation
EDSL's `run(n=X)` should:
- For OpenAI: Use native `n` parameter
- For Gemini 2.0+: Use native `candidateCount` parameter  
- For others: Fall back to current behavior

This would provide significant cost savings and performance improvements for research applications.